In [ ]:
# 1. Install Dependencies
!pip install -q fastapi uvicorn pyngrok nest_asyncio torch transformers peft bitsandbytes accelerate huggingface_hub

In [ ]:
# --- MEMORY FIX START ---
import os
# This helps with "fragmentation" errors
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# --- MEMORY FIX END ---

# 2. Imports
import torch
import gc
import uvicorn
import nest_asyncio
import os
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login


In [ ]:
# 4. The Optimized LLM Engine (Dual-Mode)
class LLMEngine:
    def __init__(self):
        self.base_model_id = "defog/llama-3-sqlcoder-8b"
        self.adapter_id = "Sourish-Kanna/CenQuery"

        print("🧹 Cleaning GPU Memory...")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        print(f"⏳ Loading Base Model: {self.base_model_id}...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )

        self.base_model = AutoModelForCausalLM.from_pretrained(
            self.base_model_id,
            device_map="auto",
            quantization_config=bnb_config,
            trust_remote_code=True
        )
        self.base_model.config.use_cache = False

        self.tokenizer = AutoTokenizer.from_pretrained(self.base_model_id)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "right"

        print(f"⏳ Downloading Adapter: {self.adapter_id}...")
        self.model = PeftModel.from_pretrained(self.base_model, self.adapter_id, is_trainable=False)
        self.model.eval()

        self.terminators = [
            self.tokenizer.eos_token_id,
            self.tokenizer.convert_tokens_to_ids("<|eot_id|>")
        ]
        print("✅ System Ready! CenQuery Brain is Online.")

    def generate(self, prompt: str, use_adapter: bool = True):
        gc.collect()
        torch.cuda.empty_cache()

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            if use_adapter:
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=300,
                    num_return_sequences=1,
                    eos_token_id=self.terminators,
                    pad_token_id=self.tokenizer.eos_token_id,
                    do_sample=False
                )
            else:
                # Bypass adapter for bare model benchmarking
                with self.model.disable_adapter():
                    outputs = self.model.generate(
                        **inputs,
                        max_new_tokens=300,
                        num_return_sequences=1,
                        eos_token_id=self.terminators,
                        pad_token_id=self.tokenizer.eos_token_id,
                        do_sample=False
                    )

        full_output = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        if prompt in full_output:
            generated_text = full_output.replace(prompt, "").strip()
        else:
            generated_text = full_output

        if "### SQL" in generated_text:
            generated_text = generated_text.split("### SQL")[-1].strip()

        clean_sql = generated_text.split("assistant")[0].split("<|start_header_id|>")[0].strip()

        if ";" in clean_sql:
            clean_sql = clean_sql.split(";")[0] + ";"

        return clean_sql

In [ ]:
# Initialize App & Engine
app = FastAPI(title="CenQuery LLM Service (Benchmarking Enabled)")
nest_asyncio.apply()
engine = LLMEngine()

In [ ]:
# 5. API Endpoints (Dual Mode)
class QueryRequest(BaseModel):
    prompt: str

@app.post("/generate/adapter")
async def generate_sql_adapter(req: QueryRequest):
    try:
        print(f"🔸 [ADAPTER] Prompt: [{req.prompt}]")
        sql = engine.generate(req.prompt, use_adapter=True)
        print(f"🔹 [ADAPTER] Generated: {sql}") # Log for debugging in Colab
        return {"sql": sql, "model": "CenQuery-Adapter"}
    except Exception as e:
        print(f"❌ Error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/generate/base")
async def generate_sql_base(req: QueryRequest):
    try:
        print(f"🔸 [BASE] Prompt: [{req.prompt}]")
        sql = engine.generate(req.prompt, use_adapter=False)
        print(f"🔹 [BASE] Generated: {sql}") # Log for debugging in Colab
        return {"sql": sql, "model": "Llama-3-SQLCoder-8B-Base"}
    except Exception as e:
        print(f"❌ Error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health", include_in_schema=False)
async def health_check():
    return {"status": "healthy", "message": "API is running on Google Colab"}

@app.get("/", include_in_schema=False)
async def root():
    return {"message": "CenQuery LLM Service (Benchmarking Enabled) is Online!"}

In [ ]:
# 6. Start Ngrok
# PASTE YOUR NGROK TOKEN BELOW (Get it from dashboard.ngrok.com)
# NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"
from google.colab import userdata
NGROK_AUTH_TOKEN = userdata.get('NGROK')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000).public_url
print(f"🚀 Service A is Online at: {public_url}")



In [ ]:
# 7. Run Server
# Apply the patch (Good practice, even if using await)
nest_asyncio.apply()

# Configure the server
config = uvicorn.Config(app, port=8000)
server = uvicorn.Server(config)

# Start the server in the existing loop
# This cell will stay "Busy" [*] - that is normal!
await server.serve()

In [ ]:
from pyngrok import ngrok

# 1. Kill all running tunnels
ngrok.kill()

print("🛑 ngrok tunnels killed. You can now restart the server.")